# Imported required libraries

In [1]:
# For Data Handling & System Utilities
import pandas as pd
import numpy as np
import random
import os

# For Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, MaxNLocator
from IPython.display import Markdown, display
from IPython.display import display, HTML

# Statistical & Distribution Analysis
from scipy.stats import gamma
import missingno as msno

# Machine Learning (Data Splitting)
from sklearn.model_selection import train_test_split

In [2]:
# Machine Learning – Data Preparation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning – Models (Classification)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Model Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    precision_recall_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay)

In [3]:
# For exporting ML model
import pickle

In [4]:
# For statistical calculation
from scipy import stats

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Importing Preprocessed Train, Test and Holdout Dataset

In [7]:
# Define the directory where the data is stored
base_path = r"C:/Users/danie/Downloads/Data Analytic/credit_loan_analysis"

try:
    # Attempt to read the specific CSV files
    # We use os.path.join for path safety
    y_train = pd.read_csv(os.path.join(base_path, 'loan_targets_train.csv'))
    y_test = pd.read_csv(os.path.join(base_path, 'loan_targets_test.csv'))
    y_holdout = pd.read_csv(os.path.join(base_path, 'loan_targets_holdout.csv'))

    X_train = pd.read_csv(os.path.join(base_path, 'loan_inputs_train.csv'))
    X_test = pd.read_csv(os.path.join(base_path, 'loan_inputs_test.csv'))
    X_holdout = pd.read_csv(os.path.join(base_path, 'loan_inputs_holdout.csv'))
    
except FileNotFoundError:
    print(f"Error: Could not find the files in '{base_path}'.")
    print("Please verify the directory path and ensure the files were exported correctly.")
except Exception as e:
    print(f"An unexpected error occurred during import: {e}")

else:
    # Success confirmation and data verification
    print("Data successfully imported.")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")
    print(f"y_holdout shape: {y_holdout.shape}")

    print(f"X_train shape: {y_train.shape}")
    print(f"X_test shape: {y_test.shape}")
    print(f"X_holdout shape: {y_holdout.shape}")

Data successfully imported.
y_train shape: (240000, 1)
y_test shape: (80000, 1)
y_holdout shape: (80000, 1)
X_train shape: (240000, 1)
X_test shape: (80000, 1)
X_holdout shape: (80000, 1)


In [8]:
X_train.head()

,purpose,int_rate,interest_band,grade,loan_to_value,collateral,regularity_of_inflows,ltv_group,ead
0,Car Purchase,8.708671,7.5-9.5,B,0.695281,Yes,Yes,Moderate,0.0
1,Small Business,11.854966,11.5-14.9,C,0.802834,Yes,Yes,High,0.0
2,Home Improvement,11.365925,9.5-11.5,C,0.560825,Yes,Yes,Low,0.0
3,Debt Consolidation,12.119646,11.5-14.9,C,0.915366,Yes,Yes,High,0.0
4,Debt Consolidation,10.079624,9.5-11.5,B,0.451274,No,Yes,Low,0.0


# Import the model

In [9]:
# Define the path
target_directory = r"C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\models"
model_path = os.path.join(target_directory, "credit_risk_PD_model.pkl")
results_path = os.path.join(target_directory, "final_scoring_results.csv")

In [10]:
# Load Model Artifacts
with open(model_path, 'rb') as file:
    loaded_artifacts = pickle.load(file)

# Extract components
logit_model = loaded_artifacts["model"]
score_factor = loaded_artifacts["factor"]
threshold = loaded_artifacts["optimal_threshold"]

In [11]:
# Load Scoring Results (holdout set)
df_final = pd.read_csv(results_path)
df_final.head()

,Probability_of_Default,Credit_Score,Prediction,Actual_Status,Rating
0,0.000977,987,Not Default,Not Default,Excellent
1,0.000987,986,Not Default,Not Default,Excellent
2,0.001673,971,Not Default,Not Default,Excellent
3,0.000626,999,Not Default,Not Default,Excellent
4,0.011219,916,Not Default,Not Default,Good


# Segmented LGD given the context and reference

| Purpose              | Est. Recovery Rate (RR) | LGD (1 − RR) | Logic                                                         |
|----------------------|------------------------|--------------|---------------------------------------------------------------|
| Home Improvement     | 45%                    | 55%          | Often backed by home equity / stable homeowners               |
| Debt Consolidation   | 25%                    | 75%          | High risk; borrower is already struggling with debt           |
| Car Purchase         | 40%                    | 60%          | Partial recovery possible through vehicle repossession        |
| Major Purchase       | 30%                    | 70%          | Depreciating assets (e.g., electronics, furniture)            |
| Small Business       | 20%                    | 80%          | Highest risk; business assets often have low resale value     |

| Loan Type                     | Typical LGD (UK/Global Standards) | Recovery Reasoning                                      |
|------------------------------|-----------------------------------|----------------------------------------------------------|
| Mortgages (Residential)      | 10% – 20%                         | High collateral value (property)                         |
| Car Loans (Secured)          | 40% – 50%                         | Depreciating asset, but repossessable                   |
| Unsecured Personal Loans     | 70% – 90%                         | No collateral; rely on debt collection agencies         |
| Credit Cards                 | 85% – 95%                         | Highest risk; usually very low recovery                 |

In [12]:
# Configuration (Business Assumptions)
## Stress scenario: reduce recovery rates by 15% during downturn
DOWNTURN_HAIRCUT = 0.15 
## Mapping loan purpose → expected recovery rate (RR)
## RR = % of money the bank can recover if default happens
purpose_recovery_mapping = {
    'Car Purchase': 0.40,
    'Home Improvement': 0.45,
    'Major Purchase': 0.30,
    'Debt Consolidation': 0.25,
    'Small Business': 0.20}

# Function to compute LGD
def compute_downturn_lgd(X_df, y_df, name="Dataset"):
    """
    Computes both Base LGD and Downturn LGD.
    Downturn LGD reflects the stressed loss during economic instability.
    """
    df = y_df.copy()
    target_col = df.columns[0]
    
    # Map Base Recovery Rate
    df['Base_RR'] = X_df['purpose'].map(purpose_recovery_mapping)
    
    # Compute Downturn Recovery Rate (ensure it doesn't go below 0)
    df['Downturn_RR'] = (df['Base_RR'] - DOWNTURN_HAIRCUT).clip(lower=0)
    
    # Calculate LGDs (Only if Default = 1)
    df['Base_LGD'] = df.apply(
        lambda row: (1 - row['Base_RR']) if row[target_col] == 1 else 0.0, axis=1)
    
    df['Downturn_LGD'] = df.apply(
        lambda row: (1 - row['Downturn_RR']) if row[target_col] == 1 else 0.0, axis=1)
    
    # Bring EAD and purpose column from X_df
    df['ead'] = X_df['ead'].values
    df['purpose'] = X_df['purpose'].values

    # Calculate Monetary Losses (EAD * LGD)
    df['Base_Loss_Amount'] = (df['ead'] * df['Base_LGD']).round(2)
    df['Downturn_Loss_Amount'] = (df['ead'] * df['Downturn_LGD']).round(2)
    
    # Compute summary metrics
    avg_base = df[df[target_col] == 1]['Base_LGD'].mean()
    avg_downturn = df[df[target_col] == 1]['Downturn_LGD'].mean()

    # Compute total loss sums
    total_base_loss = df['Base_Loss_Amount'].sum()
    total_downturn_loss = df['Downturn_Loss_Amount'].sum()

    print(f"--- {name} Downturn Analysis ---")
    print(f"Average Base LGD:      {avg_base:.2%}")
    print(f"Average Downturn LGD:  {avg_downturn:.2%}")
    print(f"Incremental Stress:    {avg_downturn - avg_base:.2%}")
    print(f"Total Base Loss:       £{total_base_loss:,.2f}")
    print(f"Total Downturn Loss:   £{total_downturn_loss:,.2f}")
    print("-" * 40)
    
    return df

In [13]:
# Compute LGD for training data
y_train_lgd_full = compute_downturn_lgd(X_train, y_train, "y_train")
# Compute LGD for test data
y_test_lgd_full = compute_downturn_lgd(X_test, y_test, "y_test")
# Compute LGD for holdout data
y_holdout_lgd_full = compute_downturn_lgd(X_holdout, y_holdout, "y_holdout")

--- y_train Downturn Analysis ---
Average Base LGD:      68.21%
Average Downturn LGD:  83.21%
Incremental Stress:    15.00%
Total Base Loss:       £363,562,051.61
Total Downturn Loss:   £447,143,449.33
----------------------------------------
--- y_test Downturn Analysis ---
Average Base LGD:      68.05%
Average Downturn LGD:  83.05%
Incremental Stress:    15.00%
Total Base Loss:       £121,314,081.70
Total Downturn Loss:   £149,241,239.49
----------------------------------------
--- y_holdout Downturn Analysis ---
Average Base LGD:      68.05%
Average Downturn LGD:  83.05%
Incremental Stress:    15.00%
Total Base Loss:       £122,760,688.10
Total Downturn Loss:   £151,098,087.32
----------------------------------------


In [14]:
# Display a comparison of the top rows
display(y_holdout_lgd_full[[y_holdout.columns[0], 'purpose', 'Base_LGD', 'Downturn_LGD', 'ead', 'Base_Loss_Amount', 'Downturn_Loss_Amount']].head(10))

,default_status,purpose,Base_LGD,Downturn_LGD,ead,Base_Loss_Amount,Downturn_Loss_Amount
0,0,Debt Consolidation,0.0,0.0,0.0,0.0,0.0
1,0,Small Business,0.0,0.0,0.0,0.0,0.0
2,0,Major Purchase,0.0,0.0,0.0,0.0,0.0
3,0,Debt Consolidation,0.0,0.0,0.0,0.0,0.0
4,0,Car Purchase,0.0,0.0,0.0,0.0,0.0
5,0,Major Purchase,0.0,0.0,0.0,0.0,0.0
6,0,Debt Consolidation,0.0,0.0,0.0,0.0,0.0
7,0,Debt Consolidation,0.0,0.0,0.0,0.0,0.0
8,0,Home Improvement,0.0,0.0,0.0,0.0,0.0
9,0,Home Improvement,0.0,0.0,0.0,0.0,0.0


# Compute Expected Loss
$$EL = PD \times LGD \times EAD$$

In [15]:
# Combine two dataframe for EL calculation
el_holdout_df = pd.concat([
    df_final.reset_index(drop=True), 
    y_holdout_lgd_full.reset_index(drop=True)
], axis=1)

# Calculate Expected Loss (EL) for each borrower (holdout set)
el_holdout_df['Expected_Loss_Base'] = (
    el_holdout_df['Probability_of_Default'] * el_holdout_df['Base_LGD'] * el_holdout_df['ead']
).round(2)

el_holdout_df['Expected_Loss_Downturn'] = (
    el_holdout_df['Probability_of_Default'] * el_holdout_df['Downturn_LGD'] * el_holdout_df['ead']
).round(2)

# Aggregate Metrics for Reporting
metrics = {
    "Total EAD": el_holdout_df['ead'].sum(),
    "Base EL": el_holdout_df['Expected_Loss_Base'].sum(),
    "Downturn EL": el_holdout_df['Expected_Loss_Downturn'].sum(),
    "Incremental Capital": el_holdout_df['Expected_Loss_Downturn'].sum() - el_holdout_df['Expected_Loss_Base'].sum()
}

print(f"--- Holdout Expected Loss (EL) Report ---")
# print(f"Total Portfolio EAD:          £{metrics['Total EAD']:,.2f}")
print(f"Base Expected Loss:           £{metrics['Base EL']:,.2f}")
print(f"Downturn Expected Loss:       £{metrics['Downturn EL']:,.2f}")
print(f"Incremental Capital Required: £{metrics['Incremental Capital']:,.2f}")
print(f"Overall Loss Rate (Base):     {(metrics['Base EL'] / metrics['Total EAD']):.2%}")
print("-" * 42)

# Display sample
display(el_holdout_df[['Credit_Score', 'Probability_of_Default', 'Base_LGD', 'ead', 'Expected_Loss_Base']].head(100))

--- Holdout Expected Loss (EL) Report ---
Base Expected Loss:           £107,160,605.64
Downturn Expected Loss:       £131,893,916.84
Incremental Capital Required: £24,733,311.20
Overall Loss Rate (Base):     56.72%
------------------------------------------


,Credit_Score,Probability_of_Default,Base_LGD,ead,Expected_Loss_Base
0,987,0.000977,0.00,0.00,0.00
1,986,0.000987,0.00,0.00,0.00
2,971,0.001673,0.00,0.00,0.00
3,999,0.000626,0.00,0.00,0.00
4,916,0.011219,0.00,0.00,0.00
...,...,...,...,...,...
95,736,0.850684,0.00,0.00,0.00
96,999,0.000355,0.00,0.00,0.00
97,916,0.010993,0.00,0.00,0.00
98,724,0.899032,0.75,18093.74,12200.14


In [16]:
# Combine two dataframe for EL calculation
el_test_df = pd.concat([
    df_final.reset_index(drop=True), 
    y_test_lgd_full.reset_index(drop=True)
], axis=1)

# Calculate Expected Loss (EL) for each borrower (holdout set)
el_test_df['Expected_Loss_Base'] = (
    el_test_df['Probability_of_Default'] * el_holdout_df['Base_LGD'] * el_holdout_df['ead']
).round(2)

el_test_df['Expected_Loss_Downturn'] = (
    el_test_df['Probability_of_Default'] * el_holdout_df['Downturn_LGD'] * el_holdout_df['ead']
).round(2)

# Aggregate Metrics for Reporting
metrics = {
    "Total EAD": el_test_df['ead'].sum(),
    "Base EL": el_test_df['Expected_Loss_Base'].sum(),
    "Downturn EL": el_test_df['Expected_Loss_Downturn'].sum(),
    "Incremental Capital": el_test_df['Expected_Loss_Downturn'].sum() - el_holdout_df['Expected_Loss_Base'].sum()
}

print(f"--- Holdout Expected Loss (EL) Report ---")
# print(f"Total Portfolio EAD:          £{metrics['Total EAD']:,.2f}")
print(f"Base Expected Loss:           £{metrics['Base EL']:,.2f}")
print(f"Downturn Expected Loss:       £{metrics['Downturn EL']:,.2f}")
print(f"Incremental Capital Required: £{metrics['Incremental Capital']:,.2f}")
print(f"Overall Loss Rate (Base):     {(metrics['Base EL'] / metrics['Total EAD']):.2%}")
print("-" * 42)

# Display sample
display(el_test_df[['Credit_Score', 'Probability_of_Default', 'Base_LGD', 'ead', 'Expected_Loss_Base']].head(100))

--- Holdout Expected Loss (EL) Report ---
Base Expected Loss:           £107,160,605.64
Downturn Expected Loss:       £131,893,916.84
Incremental Capital Required: £24,733,311.20
Overall Loss Rate (Base):     57.56%
------------------------------------------


,Credit_Score,Probability_of_Default,Base_LGD,ead,Expected_Loss_Base
0,987,0.000977,0.0,0.0,0.00
1,986,0.000987,0.0,0.0,0.00
2,971,0.001673,0.0,0.0,0.00
3,999,0.000626,0.0,0.0,0.00
4,916,0.011219,0.0,0.0,0.00
...,...,...,...,...,...
95,736,0.850684,0.0,0.0,0.00
96,999,0.000355,0.0,0.0,0.00
97,916,0.010993,0.0,0.0,0.00
98,724,0.899032,0.0,0.0,12200.14


In [17]:
# Define the index of the customer you want to look up
customer_index = 5  # Change this to any valid index

# Fetch the specific row
customer_data = el_test_df.loc[customer_index]

print(f"--- Individual EL Report for Index {customer_index} ---")
print(f"Credit Score:      {customer_data['Credit_Score']}")
print(f"EAD:               £{customer_data['ead']:,.2f}")
print(f"PD:                {customer_data['Probability_of_Default']:.2%}")
print(f"Base LGD:          {customer_data['Base_LGD']:.2%}")
print(f"Base EL:           £{customer_data['Expected_Loss_Base']:,.2f}")
print(f"Downturn EL:       £{customer_data['Expected_Loss_Downturn']:,.2f}")

--- Individual EL Report for Index 5 ---
Credit Score:      915
EAD:               £0.00
PD:                1.14%
Base LGD:          0.00%
Base EL:           £0.00
Downturn EL:       £0.00


In [18]:
def predict_and_score_visual(input_data, model_artifacts):
    """
    Returns a vibrant, dark-themed HTML dashboard including Loan Purpose.
    """
    # --- 1. MODEL LOGIC ---
    model = model_artifacts["model"]
    factor = model_artifacts["factor"]
    offset = model_artifacts["offset"]
    threshold = model_artifacts["optimal_threshold"]
    feature_names = model_artifacts["feature_names"]

    # Process Input
    df_input = pd.DataFrame([input_data]) if isinstance(input_data, dict) else input_data.copy()
    
    # Extract purpose for display before encoding
    loan_purpose = df_input['purpose'].iloc[0] if 'purpose' in df_input.columns else "N/A"
    
    df_encoded = pd.get_dummies(df_input).reindex(columns=feature_names, fill_value=0)
    
    # Calculate PD and Score
    pd_val = model.predict_proba(df_encoded)[:, 1][0]
    pd_val_clipped = np.clip(pd_val, 0.0001, 0.9999)
    log_odds = np.log(pd_val_clipped / (1 - pd_val_clipped))
    credit_score = int(offset - (factor * log_odds))
    
    # Decision
    prediction = "DEFAULT" if pd_val >= threshold else "NOT DEFAULT"

    # --- 2. COLOR DYNAMICS ---
    if credit_score >= 881:
        accent_color, rating = "#00ff88", "EXCELLENT" 
    elif credit_score >= 721:
        accent_color, rating = "#f1c40f", "FAIR / GOOD" 
    else:
        accent_color, rating = "#ff4757", "HIGH RISK" 

    pred_box_bg = "#ff4757" if prediction == "DEFAULT" else "#2ed573"

    # --- 3. HTML & CSS DESIGN ---
    html_output = f"""
    <div style="background: linear-gradient(135deg, #1e272e 0%, #2f3542 100%); 
                padding: 30px; border-radius: 20px; width: 500px; 
                color: white; font-family: 'Segoe UI', Arial, sans-serif;
                box-shadow: 0 10px 30px rgba(0,0,0,0.5); border: 1px solid #57606f;">
        
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #57606f; padding-bottom: 15px; margin-bottom: 20px;">
            <h2 style="margin: 0; font-size: 20px; color: #ced4da; letter-spacing: 1px;">CREDIT ANALYSIS</h2>
            <span style="background: #ffa502; color: #2f3542; padding: 4px 10px; border-radius: 5px; font-size: 10px; font-weight: bold;">LIVE MODEL V1.0</span>
        </div>

        <div style="display: flex; align-items: center; justify-content: space-around; margin-bottom: 25px;">
            <div style="text-align: center;">
                <div style="font-size: 11px; color: #a4b0be; text-transform: uppercase; margin-bottom: 5px;">Credit Score</div>
                <div style="font-size: 50px; font-weight: 800; color: {accent_color}; text-shadow: 0 0 15px {accent_color}55;">{credit_score}</div>
            </div>
            <div style="width: 2px; height: 60px; background: #57606f;"></div>
            <div style="text-align: center;">
                <div style="font-size: 11px; color: #a4b0be; text-transform: uppercase; margin-bottom: 5px;">Risk Rating</div>
                <div style="font-size: 22px; font-weight: 700; color: {accent_color};">{rating}</div>
            </div>
        </div>

        <div style="background: rgba(255,255,255,0.05); padding: 20px; border-radius: 15px;">
            <div style="display: flex; justify-content: space-between; margin-bottom: 10px;">
                <span style="color: #a4b0be;">Loan Purpose:</span>
                <span style="font-weight: bold; color: {accent_color};">{loan_purpose}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 10px;">
                <span style="color: #a4b0be;">Prob. of Default:</span>
                <span style="font-weight: bold; color: white;">{pd_val:.4%}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
                <span style="color: #a4b0be;">Model Threshold:</span>
                <span style="font-weight: bold; color: white;">{threshold:.2%}</span>
            </div>
            
            <div style="background: {pred_box_bg}; color: white; padding: 12px; border-radius: 8px; text-align: center; font-weight: 800; font-size: 18px; letter-spacing: 2px; box-shadow: 0 4px 10px rgba(0,0,0,0.3);">
                {prediction}
            </div>
        </div>

        <div style="margin-top: 20px; font-size: 10px; color: #747d8c; text-align: center;">
            DATA PIPELINE ENCRYPTED | ESTIMATED EAD INCLUDED IN CALCULATION
        </div>
    </div>
    """
    return display(HTML(html_output))

# 1. High Risk Sample (Low score, high rate, high LTV)
sample_poor = {
    'int_rate': 22.0, 'loan_to_value': 0.98, 'ead': 45000,
    'interest_band': '20.0+', 'grade': 'G', 'collateral': 'No',
    'regularity_of_inflows': 'No', 'ltv_group': 'Critical', 'purpose': 'Small Business'
}

# 2. Low Risk Sample (High score, low rate, low LTV)
sample_excellent = {
    'int_rate': 5.5, 'loan_to_value': 0.40, 'ead': 12000,
    'interest_band': '5.0-7.5', 'grade': 'A', 'collateral': 'Yes',
    'regularity_of_inflows': 'Yes', 'ltv_group': 'Low', 'purpose': 'Home Improvement'
}


# --- SAMPLE EXECUTION ---
print("Running Risk Assessments with Purpose...")
predict_and_score_visual(sample_excellent, loaded_artifacts)
predict_and_score_visual(sample_poor, loaded_artifacts)

Running Risk Assessments with Purpose...


In [19]:
def predict_and_score_visual(input_data, model_artifacts):
    # --- 1. MODEL LOGIC ---
    model = model_artifacts["model"]
    factor = model_artifacts["factor"]
    offset = model_artifacts["offset"]
    threshold = model_artifacts["optimal_threshold"]
    feature_names = model_artifacts["feature_names"]

    # Process Input
    df_input = pd.DataFrame([input_data]) if isinstance(input_data, dict) else input_data.copy()
    loan_purpose = df_input['purpose'].iloc[0] if 'purpose' in df_input.columns else "Other"
    ead_val = df_input['ead'].iloc[0] if 'ead' in df_input.columns else 0
    
    # --- 2. LGD MAPPING LOGIC ---
    # Mapping Purpose to LGD (1 - Recovery Rate)
    lgd_map = {
        'Home Improvement': 0.55,
        'Debt Consolidation': 0.75,
        'Car Purchase': 0.60,
        'Major Purchase': 0.70,
        'Small Business': 0.80,
        'Other': 0.70
    }
    lgd_val = lgd_map.get(loan_purpose, 0.70) # Default to 70% if purpose not found

    # Calculate PD and Score
    df_encoded = pd.get_dummies(df_input).reindex(columns=feature_names, fill_value=0)
    pd_val = model.predict_proba(df_encoded)[:, 1][0]
    pd_val_clipped = np.clip(pd_val, 0.0001, 0.9999)
    log_odds = np.log(pd_val_clipped / (1 - pd_val_clipped))
    credit_score = int(offset - (factor * log_odds))
    
    # --- 3. EXPECTED LOSS CALCULATION ---
    # EL = PD * LGD * EAD
    expected_loss = pd_val * lgd_val * ead_val

    # Decision
    prediction = "DEFAULT" if pd_val >= threshold else "NOT DEFAULT"

    # --- 4. COLOR DYNAMICS ---
    if credit_score >= 881:
        accent_color, rating = "#00ff88", "EXCELLENT" 
    elif credit_score >= 721:
        accent_color, rating = "#f1c40f", "FAIR / GOOD" 
    else:
        accent_color, rating = "#ff4757", "HIGH RISK" 

    pred_box_bg = "#ff4757" if prediction == "DEFAULT" else "#2ed573"

    # --- 5. HTML & CSS DESIGN ---
    html_output = f"""
    <div style="background: linear-gradient(135deg, #1e272e 0%, #2f3542 100%); 
                padding: 30px; border-radius: 20px; width: 500px; 
                color: white; font-family: 'Segoe UI', Arial, sans-serif;
                box-shadow: 0 10px 30px rgba(0,0,0,0.5); border: 1px solid #57606f;">
        
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #57606f; padding-bottom: 15px; margin-bottom: 20px;">
            <h2 style="margin: 0; font-size: 20px; color: #ced4da; letter-spacing: 1px;">CREDIT ANALYSIS</h2>
            <span style="background: #ffa502; color: #2f3542; padding: 4px 10px; border-radius: 5px; font-size: 10px; font-weight: bold;">EL CALC ENABLED</span>
        </div>

        <div style="display: flex; align-items: center; justify-content: space-around; margin-bottom: 25px;">
            <div style="text-align: center;">
                <div style="font-size: 11px; color: #a4b0be; text-transform: uppercase; margin-bottom: 5px;">Credit Score</div>
                <div style="font-size: 50px; font-weight: 800; color: {accent_color}; text-shadow: 0 0 15px {accent_color}55;">{credit_score}</div>
            </div>
            <div style="width: 2px; height: 60px; background: #57606f;"></div>
            <div style="text-align: center;">
                <div style="font-size: 11px; color: #a4b0be; text-transform: uppercase; margin-bottom: 5px;">Expected Loss</div>
                <div style="font-size: 22px; font-weight: 700; color: #ffffff;">£{expected_loss:,.2f}</div>
            </div>
        </div>

        <div style="background: rgba(255,255,255,0.05); padding: 20px; border-radius: 15px;">
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #a4b0be;">Loan Purpose:</span>
                <span style="font-weight: bold;">{loan_purpose} (LGD: {lgd_val:.0%})</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #a4b0be;">Exposure (EAD):</span>
                <span style="font-weight: bold; color: white;">£{ead_val:,.2f}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #a4b0be;">Prob. of Default:</span>
                <span style="font-weight: bold; color: white;">{pd_val:.4%}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
                <span style="color: #a4b0be;">Risk Rating:</span>
                <span style="font-weight: bold; color: {accent_color};">{rating}</span>
            </div>
            
            <div style="background: {pred_box_bg}; color: white; padding: 12px; border-radius: 8px; text-align: center; font-weight: 800; font-size: 18px; letter-spacing: 2px;">
                {prediction}
            </div>
        </div>
    </div>
    """
    return display(HTML(html_output))

# Sample 1: High Risk (Small Business - 80% LGD)
sample_business = {
    'int_rate': 24.5, 'loan_to_value': 0.95, 'ead': 50000,
    'interest_band': '20.0+', 'grade': 'G', 'collateral': 'No',
    'regularity_of_inflows': 'No', 'ltv_group': 'Critical', 
    'purpose': 'Small Business'
}

# Sample 2: Low Risk (Home Improvement - 55% LGD)
sample_home = {
    'int_rate': 6.2, 'loan_to_value': 0.35, 'ead': 20000,
    'interest_band': '5.0-7.5', 'grade': 'A', 'collateral': 'Yes',
    'regularity_of_inflows': 'Yes', 'ltv_group': 'Low', 
    'purpose': 'Home Improvement'
}

# Execution
print("Generating Risk Dashboards...")
predict_and_score_visual(sample_business, loaded_artifacts)
predict_and_score_visual(sample_home, loaded_artifacts)

Generating Risk Dashboards...


# Saving notebook

In [22]:
import shutil

# Define source and destination
current_notebook_name = "LGD and EL computation.ipynb" 
destination_path = r"C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\models"

# Ensure destination exists
if not os.path.exists(destination_path):
    os.makedirs(destination_path)

# Perform the copy
try:
    # This looks for the file in the current working directory
    shutil.copy(current_notebook_name, os.path.join(destination_path, current_notebook_name))
    print(f"✅ Notebook successfully copied to: {destination_path}")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{current_notebook_name}'.")
    print("👉 Make sure the filename matches exactly and you have saved (Ctrl+S) recently.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

✅ Notebook successfully copied to: C:\Users\danie\Downloads\Data Analytic\credit_loan_analysis\models
